# 04 — Pack, Update With History, Then Undo One Step

This notebook is an end-to-end **file** workflow using the real CLI:

`example_weights.f32 → pack → turn (keeps history) → peel → diff / compare`

**Peel** can mean two related things:

1. On a **single word**: reduce depth and soften high-frequency pieces.
2. On an **archive with history**: restore an older full snapshot (rollback).

Here, `turn` writes a valid history stack, so peeling one layer means “undo the last turn step.”

In [ ]:
from pathlib import Path
import sys
import json

notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(notebook_dir)) if str(notebook_dir) not in sys.path else None

from crankl_demo import ARTIFACT_DIR, prepare_demo_files, run_cli

demo = prepare_demo_files()
base_archive = ARTIFACT_DIR / "peel_base.crank"
turned_archive = ARTIFACT_DIR / "peel_turned.crank"
peeled_archive = ARTIFACT_DIR / "peel_restored.crank"

# `pack` creates the baseline archive; `turn` records one full snapshot per step.
run_cli("pack", "--input", demo.source_path, "-o", base_archive)
run_cli("turn", "--input", base_archive, "--target", demo.target_path,
        "--steps", 4, "--lr", 0.04, "-o", turned_archive)

print("\nTurned archive summary:")
turned_summary = run_cli("inspect", turned_archive, "--json", expect_json=True)

## Roll back one history layer

The turned archive holds four step snapshots. `peel --layers 1` restores the previous snapshot into a new file.

We then **diff / compare** baseline, turned, and peeled archives to check if rollback occured

In [ ]:
run_cli("peel", "--input", turned_archive, "--layers", 1, "-o", peeled_archive)

print("\nBaseline vs turned:")
run_cli("diff", base_archive, turned_archive)

print("\nTurned vs peeled:")
run_cli("diff", turned_archive, peeled_archive)

print("\nBaseline vs peeled (combined comparison):")
comparison = run_cli("compare", base_archive, peeled_archive, "--json", expect_json=True)

## Inspect the restored file

`inspect` confirms the peeled archive is still readable and prints its health metrics.

Note: today the CLI writes the peeled result **without** keeping leftover history. A future archive format (v3) is meant to make history retention clearer.

In [ ]:
print("Turned header/metrics:")
print(json.dumps(turned_summary, indent=2))
print("\nPeeled header/metrics:")
peeled_summary = run_cli("inspect", peeled_archive, "--json", expect_json=True)
print("\nBaseline-to-peeled comparison:")
print(json.dumps(comparison, indent=2))

## Combine two archive states (`bind`)

`bind` merges matching slots with the Clifford product. That is **composition**.
We bind baseline + peeled, then inspect how many slots the result has.

In [ ]:
bound_archive = ARTIFACT_DIR / "bound_states.crank"
run_cli("bind", base_archive, peeled_archive, "-o", bound_archive)
bound_summary = run_cli("inspect", bound_archive, "--json", expect_json=True)

print("\nBound archive summary:")
print(json.dumps(bound_summary, indent=2))